In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from keras.regularizers import l2

from keras.models import Sequential
from keras.layers import GRU, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

data = pd.read_csv('Training Dataset1.csv')

data['KEY_DATE'] = pd.to_datetime(
    data['KEY_DATE'],
    format='%d.%m.%Y'
)

data = data.drop_duplicates()

data['GROWTH_PERCENT'] = (
    data['GROWTH_PERCENT']
    .replace('-', np.nan)
)

data['GROWTH_PERCENT'] = (
    data['GROWTH_PERCENT']
    .ffill()
)

data['GROWTH_PERCENT'] = pd.to_numeric(
    data['GROWTH_PERCENT']
)

features = data[['AMOUNT', 'SHOCK', 'IMPACT']]
target = data[['GROWTH_PERCENT']]

training_size = int(len(data) * 0.80)

X_train_raw = features[:training_size]
X_test_raw = features[training_size:]

y_train_raw = target[:training_size]
y_test_raw = target[training_size:]



X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = X_scaler.fit_transform(X_train_raw)
X_test_scaled = X_scaler.transform(X_test_raw)

y_train_scaled = y_scaler.fit_transform(y_train_raw)
y_test_scaled = y_scaler.transform(y_test_raw)

timesteps = 480

X_train = []
y_train = []

for i in range(timesteps, len(X_train_scaled)):
    X_train.append(
        X_train_scaled[i-timesteps:i]
    )
    y_train.append(
        y_train_scaled[i]
    )

X_train = np.array(X_train)
y_train = np.array(y_train)



X_test = []
y_test = []

for i in range(timesteps, len(X_test_scaled)):
    X_test.append(
        X_test_scaled[i-timesteps:i]
    )
    y_test.append(
        y_test_scaled[i]
    )

X_test = np.array(X_test)
y_test = np.array(y_test)



regressor = Sequential()

regressor.add(
    GRU(
        units=64,
        return_sequences=True,
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )
)

regressor.add(Dropout(0.14673260028436005))

regressor.add(
    GRU(
        units=32,

    )
)

regressor.add(Dropout(0.14673260028436005))

regressor.add(Dense(1))

optimizer = Adam(
    learning_rate=0.0015477558716539422
)

regressor.compile(
    optimizer=optimizer,
    loss='mean_squared_error'
)



early_stop = EarlyStopping(
    monitor='loss',
    patience=20,
    restore_best_weights=True
)

history = regressor.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)



y_pred = regressor.predict(X_test)

y_pred = y_scaler.inverse_transform(
    y_pred
)

y_test_actual = y_scaler.inverse_transform(
    y_test.reshape(-1, 1)
)



rmse = np.sqrt(
    mean_squared_error(
        y_test_actual,
        y_pred
    )
)

mae = mean_absolute_error(
    y_test_actual,
    y_pred
)

r2 = r2_score(
    y_test_actual,
    y_pred
)

print("RMSE =", rmse)
print("MAE  =", mae)
print("R²   =", r2)

# =========================
# SAVE RESULTS
# =========================

results = pd.DataFrame({
    "Actual_Growth": y_test_actual.flatten(),
    "Predicted_Growth": y_pred.flatten()
})

results.to_csv(
    "GRU_Predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Epoch 1/200


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


175/175 ━━━━━━━━━━━━━━━━━━━━ 60s 321ms/step - loss: 0.0069
Epoch 2/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 58s 331ms/step - loss: 0.0049
Epoch 3/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 56s 317ms/step - loss: 0.0048
Epoch 4/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 81s 314ms/step - loss: 0.0048
Epoch 5/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 83s 318ms/step - loss: 0.0045
Epoch 6/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 56s 321ms/step - loss: 0.0043
Epoch 7/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 81s 320ms/step - loss: 0.0043
Epoch 8/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 81s 317ms/step - loss: 0.0042
Epoch 9/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 81s 313ms/step - loss: 0.0042
Epoch 10/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 56s 317ms/step - loss: 0.0039
Epoch 11/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 83s 326ms/step - loss: 0.0035
Epoch 12/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 81s 322ms/step - loss: 0.0032
Epoch 13/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 81s 316ms/step - loss: 0.0029
Epoch 14/200
175/175 ━━━━━━━━━━━━━━━━━━━━ 84s 329ms/step - loss: 0.0023
Epoch 15/200


In [ ]:
future_features = pd.read_csv("Testing Data 1 year.csv")

future_scaled = X_scaler.transform(
    future_features[['AMOUNT','SHOCK','IMPACT']]
)
# all_scale = X_scaler.transform(features)

# last_sequence = all_scale[-160:]

last_sequence = X_train_scaled[-timesteps:]

future_predictions = []

current_sequence = last_sequence.copy()

for i in range(80):

    x_input = current_sequence.reshape(
        1,
        timesteps,
        X_train.shape[2]
    )


    pred_scaled = regressor.predict(
        x_input,
        verbose=0
    )

    pred = y_scaler.inverse_transform(
        pred_scaled
    )[0][0]

    future_predictions.append(pred)

    next_feature_row = future_scaled[i]

    current_sequence = np.vstack([
        current_sequence[1:],
        next_feature_row
    ])

future_df = pd.DataFrame({
    "Future_Step": range(1,81),
    "Predicted_Growth": future_predictions
})

future_df.to_csv(
    "Future_80_Steps.csv",
    index=False
)

print("Future 80-step forecast saved.")

Future 80-step forecast saved.


In [ ]:
future_features = pd.read_csv("Testing data 10 years 1.csv")

future_scaled = X_scaler.transform(
    future_features[['AMOUNT','SHOCK','IMPACT']]
)

last_sequence = X_train_scaled[-timesteps:]

future_predictions = []

current_sequence = last_sequence.copy()

for i in range(800):

    x_input = current_sequence.reshape(
        1,
        timesteps,
        X_train.shape[2]
    )

    pred_scaled = regressor.predict(
        x_input,
        verbose=0
    )

    pred = y_scaler.inverse_transform(
        pred_scaled
    )[0][0]

    future_predictions.append(pred)

    next_feature_row = future_scaled[i]

    current_sequence = np.vstack([
        current_sequence[1:],
        next_feature_row
    ])

future_df = pd.DataFrame({
    "Future_Step": range(1, len(future_predictions)+1),
    "Predicted_Growth": future_predictions
})

future_df.to_csv(
    "Future_10_Years_Predictions3.csv",
    index=False
)

print("Future 10-year forecast saved.")

Future 10-year forecast saved.


In [ ]:
future_features = pd.read_csv("Testing data 10 years 1 - Copy.csv")

future_scaled = X_scaler.transform(
    future_features[['AMOUNT','SHOCK','IMPACT']]
)

last_sequence = X_train_scaled[-160:]

future_predictions = []

current_sequence = last_sequence.copy()

for i in range(646):

    x_input = current_sequence.reshape(
        1,
        timesteps,
        X_train.shape[2]
    )

    pred_scaled = regressor.predict(
        x_input,
        verbose=0
    )

    pred = y_scaler.inverse_transform(
        pred_scaled
    )[0][0]

    future_predictions.append(pred)

    next_feature_row = future_scaled[i]

    current_sequence = np.vstack([
        current_sequence[1:],
        next_feature_row
    ])

future_df = pd.DataFrame({
    "Future_Step": range(1, len(future_predictions)+1),
    "Predicted_Growth": future_predictions
})

future_df.to_csv(
    "Future_10_Years_Predictions2.csv",
    index=False
)

print("Future 10-year forecast saved.")

Future 10-year forecast saved.


In [ ]:
future_features = pd.read_csv("Testing data 10 years 1.csv")

future_scaled = X_scaler.transform(
    future_features[['AMOUNT','SHOCK','IMPACT']]
)

last_sequence = X_train_scaled[-timesteps:]

future_predictions = []

current_sequence = last_sequence.copy()

for i in range(800):

    x_input = current_sequence.reshape(
        1,
        timesteps,
        X_train.shape[2]
    )

    pred_scaled = regressor.predict(
        x_input,
        verbose=0
    )

    pred = y_scaler.inverse_transform(
        pred_scaled
    )[0][0]

    future_predictions.append(pred)

    next_feature_row = future_scaled[i]

    current_sequence = np.vstack([
        current_sequence[1:],
        next_feature_row
    ])

future_df = pd.DataFrame({
    "Future_Step": range(1, len(future_predictions)+1),
    "Predicted_Growth": future_predictions
})

future_df.to_csv(
    "Future_10_Years_Predictions(x).csv",
    index=False
)

print("Future 10-year forecast saved.")

Future 10-year forecast saved.


In [ ]:
import pandas as pd

df = pd.read_csv("Future_80_Steps.csv")

count_above_20 = (df["Predicted_Growth"] > 20).sum()

print("Values > 20:", count_above_20)

Values > 20: 29
